In [ ]:
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 17.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. Variáveis Linguísticas e Universo
km_oleo = ctrl.Antecedent(np.arange(0, 4001, 1), 'km_oleo')
folga_corrente = ctrl.Antecedent(np.arange(0, 6.1, 0.1), 'folga_corrente')
urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

# 2. Funções de Pertinência (3 termos por variável)
km_oleo['novo'] = fuzz.trapmf(km_oleo.universe, [0, 0, 1500, 2000])
km_oleo['uso_moderado'] = fuzz.trimf(km_oleo.universe, [1500, 2500, 3500])
km_oleo['vencido'] = fuzz.trapmf(km_oleo.universe, [3000, 3800, 4000, 4000])

folga_corrente['justa'] = fuzz.trapmf(folga_corrente.universe, [0, 0, 2, 3])
folga_corrente['aceitavel'] = fuzz.trimf(folga_corrente.universe, [2, 3, 4])
folga_corrente['perigosa'] = fuzz.trapmf(folga_corrente.universe, [3, 4.5, 6, 6])

urgencia['baixa'] = fuzz.trapmf(urgencia.universe, [0, 0, 25, 50])
urgencia['media'] = fuzz.trimf(urgencia.universe, [25, 50, 75])
urgencia['alta'] = fuzz.trapmf(urgencia.universe, [50, 75, 100, 100])

# 3. Base de Regras (9 Regras - Matriz Completa sem lacunas)
r1 = ctrl.Rule(km_oleo['novo'] & folga_corrente['justa'], urgencia['baixa'])
r2 = ctrl.Rule(km_oleo['novo'] & folga_corrente['aceitavel'], urgencia['baixa'])
r3 = ctrl.Rule(km_oleo['novo'] & folga_corrente['perigosa'], urgencia['media'])

r4 = ctrl.Rule(km_oleo['uso_moderado'] & folga_corrente['justa'], urgencia['baixa'])
r5 = ctrl.Rule(km_oleo['uso_moderado'] & folga_corrente['aceitavel'], urgencia['media'])
r6 = ctrl.Rule(km_oleo['uso_moderado'] & folga_corrente['perigosa'], urgencia['alta'])

r7 = ctrl.Rule(km_oleo['vencido'] & folga_corrente['justa'], urgencia['media'])
r8 = ctrl.Rule(km_oleo['vencido'] & folga_corrente['aceitavel'], urgencia['alta'])
r9 = ctrl.Rule(km_oleo['vencido'] & folga_corrente['perigosa'], urgencia['alta'])

# 4. Controlador Mamdani
sistema = ctrl.ControlSystem([r1, r2, r3, r4, r5, r6, r7, r8, r9])
sim = ctrl.ControlSystemSimulation(sistema)

# ==========================================
# 5. CASOS DE TESTE COMENTADOS
# ==========================================

# Teste 1: Condição Ideal
# Entradas: km_oleo = 500, folga_corrente = 1.0
# Interpretação: Condições perfeitas, defuzzificação deve apontar urgência baixa (~16%).
sim.input['km_oleo'] = 500
sim.input['folga_corrente'] = 1.0
sim.compute()
print(f"Teste 1 (Ideal) - Urgência calculada: {sim.output['urgencia']:.2f}%")

# Teste 2: Condição de Transição
# Entradas: km_oleo = 2500, folga_corrente = 3.0
# Interpretação: Óleo moderado e folga no limite do aceitável. O sistema funde as regras 5 e 2, gerando urgência média (~50%).
sim.input['km_oleo'] = 2500
sim.input['folga_corrente'] = 3.0
sim.compute()
print(f"Teste 2 (Transição) - Urgência calculada: {sim.output['urgencia']:.2f}%")

# Teste 3: Caso Extremo
# Entradas: km_oleo = 3900, folga_corrente = 5.5
# Interpretação: Componentes altamente degradados ativam a R9 isoladamente. Defuzzificação aponta urgência máxima (>80%).
sim.input['km_oleo'] = 3900
sim.input['folga_corrente'] = 5.5
sim.compute()
print(f"Teste 3 (Extremo) - Urgência calculada: {sim.output['urgencia']:.2f}%")

Teste 1 (Ideal) - Urgência calculada: 19.44%
Teste 2 (Transição) - Urgência calculada: 50.00%
Teste 3 (Extremo) - Urgência calculada: 80.56%
